In [2]:
import shutil
import os
#print(len(os.listdir("refined_verified_labels_for_yolo_segmentation")))

#print(len(os.listdir("tiles_jpg_final_unzipped")))

In [3]:
label_dir = "refined_verified_labels_for_yolo_segmentation"
image_dir = "tiles_jpg_final_unzipped"
base = "/home3/zpdv81/YOLO_Segmentation_dataset_refined_label_checkerboard_spliting"

# Create YOLO folder structure

for split in ["train","val"]:
    os.makedirs(os.path.join(base,"images",split) , exist_ok = True)
    os.makedirs(os.path.join(base,"labels",split) , exist_ok = True)
    
    
# assign a tile to train or validation based checkerboard spliting 
def get_split(tile_name):
    parts = tile_name.split("_")
    
    if len(parts)<3:
        raise ValueError(f"unexpected tile filename: {tile_name}")
    

    x,y = int(parts[1]),int(parts[2])
    tile_size = 512
    grid_x , grid_y = x// tile_size , y//tile_size
    block_x , block_y = grid_x // 2 , grid_y //2
    
    if (block_x + block_y)% 5 == 0 :
        return "val"
    else:
        return "train"
    
train_count = 0
val_count = 0

# copy each image and its matching yolo label file

for label_file in os.listdir(label_dir):
    
    if not label_file.endswith(".txt"):  # only process yolo text label files
        continue
        
    # remove .txt to get matching image file name
    tile_name = os.path.splitext(label_file)[0]
    image_path = os.path.join(image_dir , tile_name + ".jpg")
    
    # skip labels without images matching
    if not os.path.exists(image_path):
        print(f" Missing image for : {label_file}")
        continue
    
    # check whether this tile belongs in train or validation
    split = get_split(tile_name)
    
    # source paths
    label_path = os.path.join(label_dir , label_file)
    
    
    # path destination
    output_label_path = os.path.join(base,'labels',split,label_file)
    output_image_path = os.path.join(base,'images' , split , tile_name + ".jpg")
    
    # copy files
    shutil.copy(label_path , output_label_path)
    shutil.copy(image_path , output_image_path)
    
    if split == "train":
        train_count +=1
    else:
        val_count +=1
    
    
print(f" Train images : {train_count}")
print(f" Validation images : {val_count}")

total = train_count + val_count
print(f" Train percentage : {train_count/total * 100:.2f}%")
print(f" Validation percentage : {val_count/total *100:.2f}%")

# create YOLO data.yaml file
data_yaml_content = f"""train : {base}/images/train
val: {base}/images/val

nc: 1
names:
 0: ridge_and_furrow
"""

yaml_path = os.path.join(base,"data.yaml")

with open(yaml_path,"w") as file:
    file.write(data_yaml_content)
    
print(f"data.yaml saved to : {yaml_path}")


 Train images : 817
 Validation images : 207
 Train percentage : 79.79%
 Validation percentage : 20.21%
data.yaml saved to : /home3/zpdv81/YOLO_Segmentation_dataset_refined_label_checkerboard_spliting/data.yaml


In [15]:
training_script = f'''
from ultralytics import YOLO

# load pre trained yolo segment model

model = YOLO("yolov8n-seg.pt")

#train the model
results = model.train(
        data = "{base}/data.yaml",
        epochs = 10,
        imgsz = 640,
        batch = 16,
        device = 0,
        workers = 2,
        scale = 0.9,
        project = "/home3/zpdv81/yolo_runs_refined_checkboard",
        name = "rnf_refined_yolo_segmentation_checkboard_sanity_check",
        )
        '''
# save the training script
with open("/home3/zpdv81/train_confirm.py","w")as f:
    f.write(training_script)
print("Script written")

Script written


In [16]:
!/home3/zpdv81/yolo_venv/bin/python /home3/zpdv81/train_confirm.py

New https://pypi.org/project/ultralytics/8.4.118 available 😃 Update with 'pip install -U ultralytics'
Ultralytics 8.4.110 🚀 Python-3.8.10 torch-2.4.1+cu121 CUDA:0 (NVIDIA A100 80GB PCIe MIG 1g.10gb, 9728MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, channels_last=False, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, cls_remap=True, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/home3/zpdv81/YOLO_Segmentation_dataset_refined_label_checkboard_spliting/data.yaml, degrees=0.0, deterministic=True, device=0, dfl=1.5, dgrad=0.5, dis=6.0, distill_model=None, dlam=1.0, dlog=1.0, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=10, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, iou=0.7, keras=False, kobj=1.0, line_wid

In [19]:
training_script = f'''
from ultralytics import YOLO

# load pre trained yolo segment model

model = YOLO("yolov8n-seg.pt")

#train the model
results = model.train(
        data = "{base}/data.yaml",
        epochs = 100,
        imgsz = 640,
        batch = 16,
        device = 0,
        workers = 2,
        scale = 0.9,
        project = "/home3/zpdv81/yolo_runs_refined_checkboard",
        name = "rnf_refined_yolo_segmentation_checkboard_2",
        )
        '''
# save the training script
with open("/home3/zpdv81/train_confirm.py","w")as f:
    f.write(training_script)
print("Script written")

Script written


In [20]:
!/home3/zpdv81/yolo_venv/bin/python /home3/zpdv81/train_confirm.py

New https://pypi.org/project/ultralytics/8.4.118 available 😃 Update with 'pip install -U ultralytics'
Ultralytics 8.4.110 🚀 Python-3.8.10 torch-2.4.1+cu121 CUDA:0 (NVIDIA A100 80GB PCIe MIG 1g.10gb, 9728MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, channels_last=False, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, cls_remap=True, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/home3/zpdv81/YOLO_Segmentation_dataset_refined_label_checkboard_spliting/data.yaml, degrees=0.0, deterministic=True, device=0, dfl=1.5, dgrad=0.5, dis=6.0, distill_model=None, dlam=1.0, dlog=1.0, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=100, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, iou=0.7, keras=False, kobj=1.0, line_wi